In [21]:
# Importing required libraries

%pip install pandas numpy
%pip install matplotlib
import pandas as pd
print(pd.__version__)
import numpy as np
import matplotlib.pyplot as plt
import re

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
3.0.3



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [22]:
#loading dataset
file_path = "providers_data.csv"
df = pd.read_csv(file_path)

In [23]:
#dataset information

print("First 5 rows :")
print(df.head())
print(f"Total Rows: {df.shape[0]} and Total Columns: {df.shape[1]}")

print("\nData types of columns")
print(df.dtypes)

First 5 rows :
   Provider_ID                         Name           Type  \
0            1             Gonzales-Cochran    Supermarket   
1            2  Nielsen, Johnson and Fuller  Grocery Store   
2            3                 Miller-Black    Supermarket   
3            4   Clark, Prince and Williams  Grocery Store   
4            5               Coleman-Farley  Grocery Store   

                                             Address            City  \
0  74347 Christopher Extensions\nAndreamouth, OK ...     New Jessica   
1           91228 Hanson Stream\nWelchtown, OR 27136     East Sheena   
2  561 Martinez Point Suite 507\nGuzmanchester, W...  Lake Jesusview   
3     467 Bell Trail Suite 409\nPort Jesus, IA 61188     Mendezmouth   
4  078 Matthew Creek Apt. 319\nSaraborough, MA 53978   Valentineside   

                Contact  
0       +1-600-220-0480  
1  +1-925-283-8901x6297  
2      001-517-295-2206  
3      556.944.8935x401  
4          193.714.6577  
Total Rows: 1000 and To

In [24]:
#missing values

print("\nMissing values per column :")
missing_counts = df.isna().sum()
print(missing_counts)


Missing values per column :
Provider_ID    0
Name           0
Type           0
Address        0
City           0
Contact        0
dtype: int64


In [25]:
#duplicate rows

print(f"Duplicate Rows Count : {df.duplicated().sum()}")

Duplicate Rows Count : 0


In [26]:
#understanding variables

print("\nColumn Summary (Numeric type Only) :")

numeric_cols = df.select_dtypes(include=[np.number]).columns

if len(numeric_cols) > 0:
    print(df[numeric_cols].describe())
else:
    print("Dataset doesn't have any numeric type column")


Column Summary (Numeric type Only) :
       Provider_ID
count  1000.000000
mean    500.500000
std     288.819436
min       1.000000
25%     250.750000
50%     500.500000
75%     750.250000
max    1000.000000


In [27]:
# Handle datatypes
df['Provider_ID'] = df['Provider_ID'].astype('int64')

df['Name'] = df['Name'].astype('string')

df['Type'] = df['Type'].astype('category')

df['Address'] = df['Address'].astype('string')

df['City'] = df['City'].astype('string')

df['Contact'] = df['Contact'].astype('string')

print("\nUpdated Data Types:")
print(df.dtypes)


Updated Data Types:
Provider_ID       int64
Name             string
Type           category
Address          string
City             string
Contact          string
dtype: object


In [28]:
#outliers

Q1 = df['Provider_ID'].quantile(0.25)
Q3 = df['Provider_ID'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['Provider_ID'] < lower_bound) |
    (df['Provider_ID'] > upper_bound)
]

print("Number of outliers:", len(outliers))
print(outliers)

Number of outliers: 0
Empty DataFrame
Columns: [Provider_ID, Name, Type, Address, City, Contact]
Index: []


In [30]:
# Clean contact numbers
def clean_contact(contact):
    if pd.isna(contact):
        return None

    contact = str(contact)

    # Remove extension (x1234)
    contact = re.sub(r'x\d+', '', contact, flags=re.IGNORECASE)

    # Keep only digits
    contact = re.sub(r'\D', '', contact)

    return contact

df['Contact'] = df['Contact'].apply(clean_contact)

print(df[['Contact']].head())

# Count digits
df['Contact_Length'] = df['Contact'].str.len()

# Find suspicious numbers
invalid_contacts = df[
    (df['Contact_Length'] < 10) |
    (df['Contact_Length'] > 15)
]

print("Invalid Contacts:")
print(invalid_contacts[['Provider_ID', 'Contact']])

         Contact
0    16002200480
1    19252838901
2  0015172952206
3     5569448935
4     1937146577
Invalid Contacts:
Empty DataFrame
Columns: [Provider_ID, Contact]
Index: []


In [31]:
#create new file of cleaned data
df.to_csv("Providers_Data_Cleaned.csv", index=False)